# Student Performance Predictor — EDA & Modeling
This notebook walks through the full ML pipeline:
1. Data exploration and visualization
2. Feature engineering
3. Model training (regression + classification)
4. Evaluation and interpretation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '../src')

plt.style.use('dark_background')
sns.set_palette('husl')
%matplotlib inline
print('Libraries loaded ')

## 1. Generate and Load Data

In [ ]:
from data_generator import generate_dataset
df = generate_dataset(n_samples=2000)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. EDA — Distributions and Correlations

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = ['study_hours_per_day','attendance_rate','sleep_hours',
            'assignments_completed','previous_gpa','extracurricular_hours',
            'tutoring_sessions','exam_score']
for ax, feat in zip(axes.flat, features):
    ax.hist(df[feat], bins=30, edgecolor='none', alpha=0.8)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Grade distribution
fig, ax = plt.subplots(figsize=(7, 4))
grade_order = ['A','B','C','D','F']
counts = df['grade'].value_counts().reindex(grade_order)
bars = ax.bar(grade_order, counts.values, color=['#6c63ff','#00d4aa','#ffb347','#ff7f7f','#ff5e7e'])
ax.bar_label(bars)
ax.set_title('Grade Distribution', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df.drop(columns=['grade']).corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with exam score
corr_with_target = corr['exam_score'].drop('exam_score').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
corr_with_target.plot(kind='barh', ax=ax, color=['#00d4aa' if x > 0 else '#ff5e7e' for x in corr_with_target.values])
ax.set_title('Correlation with Exam Score', fontsize=13)
ax.axvline(0, color='white', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: study hours vs exam score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['study_hours_per_day'], df['exam_score'], alpha=0.4, s=20, c='#6c63ff')
axes[0].set_xlabel('Study Hours / Day')
axes[0].set_ylabel('Exam Score')
axes[0].set_title('Study Hours vs Exam Score')

axes[1].scatter(df['attendance_rate'], df['exam_score'], alpha=0.4, s=20, c='#00d4aa')
axes[1].set_xlabel('Attendance Rate (%)')
axes[1].set_title('Attendance vs Exam Score')

plt.tight_layout()
plt.show()

## 3. Preprocessing and Feature Engineering

In [ ]:
# Save the generated data first
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/students.csv', index=False)

from preprocess import load_and_preprocess, FEATURE_COLS
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test, scaler, le = \
    load_and_preprocess('../data/raw/students.csv')

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## 4. Regression — Predict Exact Score

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

models_reg = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
}

results_reg = []
for name, model in models_reg.items():
    model.fit(X_train, y_reg_train)
    preds = model.predict(X_test)
    r2   = r2_score(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    mae  = mean_absolute_error(y_reg_test, preds)
    results_reg.append({'Model': name, 'R²': round(r2,4), 'RMSE': round(rmse,2), 'MAE': round(mae,2)})
    print(f'{name:25s} | R²={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}')

pd.DataFrame(results_reg)

In [ ]:
# Actual vs Predicted plot (Random Forest)
rf_reg = models_reg['Random Forest']
preds = rf_reg.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_reg_test, preds, alpha=0.5, s=20, c='#6c63ff')
ax.plot([0,100],[0,100], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Score')
ax.set_ylabel('Predicted Score')
ax.set_title('Actual vs Predicted Exam Score (Random Forest)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Classification — Predict Grade

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

models_clf = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest Clf':   RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
}

for name, model in models_clf.items():
    model.fit(X_train, y_clf_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_clf_test, preds)
    print(f'{name}: Accuracy = {acc:.4f}')
    print(classification_report(y_clf_test, preds, target_names=le.classes_))
    print('─'*50)

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
rf_clf = models_clf['Random Forest Clf']
preds  = rf_clf.predict(X_test)
cm = confusion_matrix(y_clf_test, preds)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted Grade')
ax.set_ylabel('Actual Grade')
ax.set_title('Confusion Matrix — Random Forest Classifier')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importances = rf_reg.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([FEATURE_COLS[i] for i in indices], importances[indices], color='#6c63ff')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance — Random Forest Regressor')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Sample Prediction

In [ ]:
# Train and save full models to disk, then test the predict module
import subprocess
result = subprocess.run(['python', '../src/train.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
from predict import predict

sample = {
    'study_hours_per_day':    5,
    'attendance_rate':        85,
    'sleep_hours':            7,
    'assignments_completed':  90,
    'previous_gpa':           3.2,
    'extracurricular_hours':  2,
    'tutoring_sessions':      3,
    'parent_education_level': 2,
}

result = predict(sample)
print('\n Prediction Result:')
for k, v in result.items():
    print(f'  {k}: {v}')